# A&R Analyst Portfolio Project
### Billboard Hit Analysis & Emerging Artist Scouting

**Prepared by:** MANAV DESAi

This analysis explores 60 years of Billboard Hot 100 chart data combined with Spotify audio features to understand what characterizes commercial hit songs — and how those characteristics have evolved over time. These findings inform a data-driven approach to scouting emerging talent.

---

## 1. Data Acquisition
Sourcing historical Billboard Hot 100 chart data and matching Spotify audio features from Kaggle.

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thedevastator/billboard-hot-100-audio-features")

print("Path to dataset files:", path)

100%|██████████| 13.4M/13.4M [00:01<00:00, 9.16MB/s]

Extracting files...
Path to dataset files: /Users/manavdesai/.cache/kagglehub/datasets/thedevastator/billboard-hot-100-audio-features/versions/1


In [2]:
import os

# List what's actually inside the downloaded folder
files = os.listdir(path)
print(files)


['Hot Stuff.csv', 'Hot 100 Audio Features.csv']


## 2. Loading & Inspecting the Data
Two files were provided: chart-performance history and Spotify audio features. Both are loaded into pandas for inspection before merging.

In [60]:
# Load both datasets into DataFrames for inspection
import pandas as pd

chart_history = pd.read_csv(os.path.join(path, "Hot Stuff.csv"))
audio_features = pd.read_csv(os.path.join(path, "Hot 100 Audio Features.csv"))

In [61]:
from IPython.display import display

print("Chart history shape:", chart_history.shape)
print("Audio features shape:", audio_features.shape)

print("\nChart history preview:")
display(chart_history.head())

print("\nAudio features preview:")
display(audio_features.head())

Chart history shape: (327895, 11)
Audio features shape: (29503, 23)

Chart history preview:


,index,url,WeekID,Week Position,Song,Performer,SongID,Instance,Previous Week Position,Peak Position,Weeks on Chart
0,0,http://www.billboard.com/charts/hot-100/1965-0...,7/17/1965,34,Don't Just Stand There,Patty Duke,Don't Just Stand TherePatty Duke,1,45.0,34,4
1,1,http://www.billboard.com/charts/hot-100/1965-0...,7/24/1965,22,Don't Just Stand There,Patty Duke,Don't Just Stand TherePatty Duke,1,34.0,22,5
2,2,http://www.billboard.com/charts/hot-100/1965-0...,7/31/1965,14,Don't Just Stand There,Patty Duke,Don't Just Stand TherePatty Duke,1,22.0,14,6
3,3,http://www.billboard.com/charts/hot-100/1965-0...,8/7/1965,10,Don't Just Stand There,Patty Duke,Don't Just Stand TherePatty Duke,1,14.0,10,7
4,4,http://www.billboard.com/charts/hot-100/1965-0...,8/14/1965,8,Don't Just Stand There,Patty Duke,Don't Just Stand TherePatty Duke,1,10.0,8,8



Audio features preview:


,index,SongID,Performer,Song,spotify_genre,spotify_track_id,spotify_track_preview_url,spotify_track_duration_ms,spotify_track_explicit,spotify_track_album,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,spotify_track_popularity
0,0,-twistin'-White Silver SandsBill Black's Combo,Bill Black's Combo,-twistin'-White Silver Sands,[],NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,¿Dònde Està Santa Claus? (Where Is Santa Claus...,Augie Rios,¿Dònde Està Santa Claus? (Where Is Santa Claus?),['novelty'],NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,......And Roses And RosesAndy Williams,Andy Williams,......And Roses And Roses,"['adult standards', 'brill building pop', 'eas...",3tvqPPpXyIgKrm4PR9HCf0,https://p.scdn.co/mp3-preview/cef4883cfd1e0e53...,166106.0,False,The Essential Andy Williams,...,-14.063,1.0,0.0315,0.91100,0.000267,0.112,0.150,83.969,4.0,38.0
3,3,...And Then There Were DrumsSandy Nelson,Sandy Nelson,...And Then There Were Drums,"['rock-and-roll', 'space age pop', 'surf music']",1fHHq3qHU8wpRKHzhojZ4a,NaN,172066.0,False,Compelling Percussion,...,-17.278,0.0,0.0361,0.00256,0.745000,0.145,0.801,121.962,4.0,11.0
4,4,...Baby One More TimeBritney Spears,Britney Spears,...Baby One More Time,"['dance pop', 'pop', 'post-teen pop']",3MjUtNVVq3C8Fn0MP3zhXa,https://p.scdn.co/mp3-preview/da2134a161f1cb34...,211066.0,False,...Baby One More Time (Digital Deluxe Version),...,-5.745,0.0,0.0307,0.20200,0.000131,0.443,0.907,92.960,4.0,77.0


In [62]:
# Confirm both datasets share a common key (SongID) that we can merge on
print("Chart history columns:")
print(chart_history.columns.tolist())

print("\nAudio features columns:")
print(audio_features.columns.tolist())

Chart history columns:
['index', 'url', 'WeekID', 'Week Position', 'Song', 'Performer', 'SongID', 'Instance', 'Previous Week Position', 'Peak Position', 'Weeks on Chart']

Audio features columns:
['index', 'SongID', 'Performer', 'Song', 'spotify_genre', 'spotify_track_id', 'spotify_track_preview_url', 'spotify_track_duration_ms', 'spotify_track_explicit', 'spotify_track_album', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'spotify_track_popularity']


## 3. Data Quality Check
Before merging, checking what proportion of songs have missing Spotify audio features (expected for older, pre-streaming-era songs).

In [63]:
# Count how many songs are missing audio feature data (likely older/pre-Spotify-era tracks)
missing_count = audio_features['danceability'].isna().sum()
total_count = len(audio_features)
print(f"Missing audio features: {missing_count} out of {total_count} songs ({missing_count/total_count:.1%})")


Missing audio features: 5169 out of 29503 songs (17.5%)


## 4. Merging the Datasets

Before merging, checking that the join key (`SongID`) is unique in the audio features table — duplicate keys would inflate row counts during the merge.

In [64]:
# Check for duplicate SongIDs, which would cause row duplication during merge
dupe_count = audio_features['SongID'].duplicated().sum()
print(f"Duplicate SongIDs found: {dupe_count}")


Duplicate SongIDs found: 117


In [65]:
# Remove duplicate SongIDs, keeping the first occurrence, so each song matches exactly once during merge
audio_features_clean = audio_features.drop_duplicates(subset='SongID')

In [66]:
# Merge chart history with the cleaned audio features table
# Using a left join to preserve all chart history rows, even for songs with no audio feature match
merged_df = pd.merge(chart_history, audio_features_clean, on='SongID', how='left')
print(merged_df.shape)

(327895, 33)


The merged dataset preserves all 327,895 original chart-history rows (one row per song per week charted), now enriched with Spotify audio features where available.

In [67]:
import os

# Save the merged dataset to disk so it doesn't need to be re-downloaded/re-merged on every run
os.makedirs('data', exist_ok=True)
merged_df.to_csv('data/billboard_merged.csv', index=False)

print("Saved to data/billboard_merged.csv")


Saved to data/billboard_merged.csv


## 5. Database Setup

Loading the merged dataset into a SQL database to enable structured querying — reflecting how this analysis would scale in a production environment.

In [68]:
from sqlalchemy import create_engine

# Create a SQLite database file to hold our merged dataset
engine = create_engine('sqlite:///data/ar_project.db')

### Cleaning Up Column Names
The merge created duplicate columns (e.g. `Song_x`/`Song_y`) since both source tables had overlapping field names. Removing redundant columns and restoring clean names.

In [25]:
merged_df.columns.tolist()

['index_x',
 'url',
 'WeekID',
 'Week Position',
 'Song_x',
 'Performer_x',
 'SongID',
 'Instance',
 'Previous Week Position',
 'Peak Position',
 'Weeks on Chart',
 'index_y',
 'Performer_y',
 'Song_y',
 'spotify_genre',
 'spotify_track_id',
 'spotify_track_preview_url',
 'spotify_track_duration_ms',
 'spotify_track_explicit',
 'spotify_track_album',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'time_signature',
 'spotify_track_popularity']

In [69]:
# Drop redundant duplicate columns created by the merge, keeping one clean version of each
merged_df = merged_df.drop(columns=['index_x', 'index_y', 'Song_y', 'Performer_y'])

In [70]:
# Rename the remaining _x columns back to their original clean names
merged_df = merged_df.rename(columns={'Song_x': 'Song', 'Performer_x': 'Performer'})

In [71]:
# Confirm the column cleanup worked as expected
print(merged_df.columns.tolist())


['url', 'WeekID', 'Week Position', 'Song', 'Performer', 'SongID', 'Instance', 'Previous Week Position', 'Peak Position', 'Weeks on Chart', 'spotify_genre', 'spotify_track_id', 'spotify_track_preview_url', 'spotify_track_duration_ms', 'spotify_track_explicit', 'spotify_track_album', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'spotify_track_popularity']


In [72]:
# Save the cleaned dataset to the database (this is the authoritative save — column names are now clean)
merged_df.to_sql('billboard_history', con=engine, if_exists='replace', index=False)

327895

In [73]:
# Confirm the table saved correctly by querying it back
test_query = pd.read_sql("SELECT * FROM billboard_history LIMIT 5", con=engine)
display(test_query)

,url,WeekID,Week Position,Song,Performer,SongID,Instance,Previous Week Position,Peak Position,Weeks on Chart,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,spotify_track_popularity
0,http://www.billboard.com/charts/hot-100/1965-0...,7/17/1965,34,Don't Just Stand There,Patty Duke,Don't Just Stand TherePatty Duke,1,45.0,34,4,...,-15.044,1.0,0.0298,0.61,0.000077,0.1,0.568,82.331,3.0,21.0
1,http://www.billboard.com/charts/hot-100/1965-0...,7/24/1965,22,Don't Just Stand There,Patty Duke,Don't Just Stand TherePatty Duke,1,34.0,22,5,...,-15.044,1.0,0.0298,0.61,0.000077,0.1,0.568,82.331,3.0,21.0
2,http://www.billboard.com/charts/hot-100/1965-0...,7/31/1965,14,Don't Just Stand There,Patty Duke,Don't Just Stand TherePatty Duke,1,22.0,14,6,...,-15.044,1.0,0.0298,0.61,0.000077,0.1,0.568,82.331,3.0,21.0
3,http://www.billboard.com/charts/hot-100/1965-0...,8/7/1965,10,Don't Just Stand There,Patty Duke,Don't Just Stand TherePatty Duke,1,14.0,10,7,...,-15.044,1.0,0.0298,0.61,0.000077,0.1,0.568,82.331,3.0,21.0
4,http://www.billboard.com/charts/hot-100/1965-0...,8/14/1965,8,Don't Just Stand There,Patty Duke,Don't Just Stand TherePatty Duke,1,10.0,8,8,...,-15.044,1.0,0.0298,0.61,0.000077,0.1,0.568,82.331,3.0,21.0


## 6. What Makes a Hit? Historical Analysis

**Key question:** Do audio characteristics (danceability, energy, tempo, positivity) meaningfully differ between Top 10 hits and songs that only charted modestly?

Starting with a simple sanity-check query before moving into deeper aggregation.

In [75]:
# Sanity check: pull a sample of songs that reached #1, to confirm filtering works correctly
query1 = '''
SELECT Song, Performer, "Peak Position", "Weeks on Chart"
FROM billboard_history
WHERE "Peak Position" = 1
LIMIT 10
'''

result1 = pd.read_sql(query1, con = engine)

print(result1)

                                                Song  \
0                            Don't Leave Me This Way   
1                            Don't Leave Me This Way   
2                            Don't Leave Me This Way   
3                            Don't Leave Me This Way   
4                            Don't Leave Me This Way   
5                            Don't Leave Me This Way   
6                                   Poor Little Fool   
7                                      One Sweet Day   
8  Candle In The Wind 1997/Something About The Wa...   
9                                Do I Make You Proud   

                    Performer  Peak Position  Weeks on Chart  
0              Thelma Houston              1              19  
1              Thelma Houston              1              20  
2              Thelma Houston              1              21  
3              Thelma Houston              1              22  
4              Thelma Houston              1              23  
5    

### Aggregating to One Row Per Song
Since the table has one row per song *per week* charted, we use `GROUP BY` to collapse this down to one row per song for the analyses that follow.

In [76]:
# Find the longest-running #1 hits by collapsing week-by-week rows into one row per song
query2 = """
SELECT Song, Performer, MAX("Weeks on Chart") AS total_weeks_charted
FROM billboard_history
WHERE "Peak Position" = 1
GROUP BY Song, Performer
ORDER BY total_weeks_charted DESC
LIMIT 10
"""

result2 = pd.read_sql(query2, con=engine)
display(result2)

,Song,Performer,total_weeks_charted
0,Party Rock Anthem,LMFAO Featuring Lauren Bennett & GoonRock,68
1,Rolling In The Deep,Adele,65
2,Circles,Post Malone,61
3,Macarena (Bayside Boys Mix),Los Del Rio,60
4,All Of Me,John Legend,59
5,Somebody That I Used To Know,Gotye Featuring Kimbra,59
6,Shape Of You,Ed Sheeran,58
7,Smooth,Santana Featuring Rob Thomas,58
8,Dark Horse,Katy Perry Featuring Juicy J,57
9,I Gotta Feeling,The Black Eyed Peas,56


### Correcting for a Sampling Bias

A naive average across all chart-week rows would over-weight songs that charted for many weeks (e.g. a 68-week hit counts 68 times, while a 2-week hit counts twice). To give every song an equal vote, we first collapse to one row per unique song before calculating tier averages.

In [77]:
# Build and test a query that collapses to one row per unique song, preserving peak position and audio features
inner_query = """
SELECT SongID, MIN("Peak Position") AS highest_position, danceability, energy, tempo, valence
FROM billboard_history
WHERE danceability IS NOT NULL
GROUP BY SongID, danceability, energy, tempo, valence
"""

test = pd.read_sql(inner_query, con=engine)
print("Unique songs with audio features:", test.shape[0])
display(test.head())

Unique songs with audio features: 24224


,SongID,highest_position,danceability,energy,tempo,valence
0,"""B"" GirlsYoung And Restless",54,0.615,0.497,193.762,0.769
1,"""Cherry Cherry"" from Hot August NightNeil Diamond",31,0.340,0.948,172.349,0.604
2,#1 Dee JayGoody Goody,82,0.859,0.376,127.202,0.902
3,#1Nelly,22,0.792,0.600,89.985,0.463
4,#9 DreamJohn Lennon,9,0.406,0.597,115.474,0.478


### 6a. All-Time (1965–Present)

In [81]:
# Compare average audio features across success tiers, using the deduplicated song-level data
query_final = """
SELECT 
    CASE 
        WHEN highest_position <= 10 THEN 'Top 10'
        WHEN highest_position <= 40 THEN 'Top 40'
        ELSE 'Charted Outside Top 40'
    END AS success_tier,
    AVG(danceability) AS avg_danceability,
    AVG(energy) AS avg_energy,
    AVG(tempo) AS avg_tempo,
    AVG(valence) AS avg_valence,
    COUNT(*) AS num_songs
FROM (
    SELECT 
        SongID,
        MIN("Peak Position") AS highest_position,
        danceability,
        energy,
        tempo,
        valence
    FROM billboard_history
    WHERE danceability IS NOT NULL
    GROUP BY SongID, danceability, energy, tempo, valence
) AS unique_songs
GROUP BY success_tier
ORDER BY avg_danceability DESC
"""

result_final = pd.read_sql(query_final, con=engine)
display(result_final)

,success_tier,avg_danceability,avg_energy,avg_tempo,avg_valence,num_songs
0,Top 10,0.617060,0.609917,119.302363,0.624523,4480
1,Top 40,0.595877,0.617801,120.317905,0.605631,7099
2,Charted Outside Top 40,0.595509,0.621071,120.614074,0.592048,12645


**Takeaway:** Across 60 years of chart history, Top 10 hits average modestly higher danceability (0.617) and valence (0.625) than songs that charted outside the Top 40 (0.596 and 0.592, respectively). Energy and tempo show minimal variation across tiers, suggesting loudness/intensity has not historically been a strong differentiator of chart success.

### 6b. 2010 Onward

In [82]:
# Repeat the same tier analysis, restricted to songs charting from 2010 onward
query_2010 = """
SELECT 
    CASE 
        WHEN highest_position <= 10 THEN 'Top 10'
        WHEN highest_position <= 40 THEN 'Top 40'
        ELSE 'Charted outside top 40'
    END AS success_tier,
    AVG(danceability) AS avg_danceability,
    AVG(energy) AS avg_energy,
    AVG(tempo) AS avg_tempo,
    AVG(valence) AS avg_valence,
    COUNT(*) AS num_songs
FROM (
    SELECT 
        SongID,
        MIN("Peak Position") AS highest_position,
        danceability,
        energy,
        tempo,
        valence
    FROM billboard_history
    WHERE danceability IS NOT NULL
    AND CAST(SUBSTR(WeekID, -4) AS INTEGER) >= 2010
    GROUP BY SongID, danceability, energy, tempo, valence
) AS unique_songs
GROUP BY success_tier
ORDER BY avg_danceability DESC
"""

result_2010 = pd.read_sql(query_2010, con=engine)
display(result_2010)

,success_tier,avg_danceability,avg_energy,avg_tempo,avg_valence,num_songs
0,Top 10,0.672339,0.670513,121.515607,0.518644,563
1,Top 40,0.642263,0.661425,122.596040,0.476206,1261
2,Charted outside top 40,0.633286,0.661568,124.015812,0.470284,3079


**Takeaway:** Since 2010, the danceability gap between Top 10 hits (0.672) and lower-charting songs (0.633) has widened compared to the all-time average — suggesting danceability has become a *stronger* predictor of success in the streaming era. Meanwhile, valence has dropped across all tiers relative to historical norms, consistent with a broader industry shift toward moodier mainstream production.

### 6c. 2020 Onward

In [83]:
# Repeat the same tier analysis, restricted to songs charting from 2020 onward
query_2020 = """
SELECT 
    CASE 
        WHEN highest_position <= 10 THEN 'Top 10'
        WHEN highest_position <= 40 THEN 'Top 40'
        ELSE 'Charted outside top 40'
    END AS success_tier,
    AVG(danceability) AS avg_danceability,
    AVG(energy) AS avg_energy,
    AVG(tempo) AS avg_tempo,
    AVG(valence) AS avg_valence,
    COUNT(*) AS num_songs
FROM (
    SELECT 
        SongID,
        MIN("Peak Position") AS highest_position,
        danceability,
        energy,
        tempo,
        valence
    FROM billboard_history
    WHERE danceability IS NOT NULL
    AND CAST(SUBSTR(WeekID, -4) AS INTEGER) >= 2020
    GROUP BY SongID, danceability, energy, tempo, valence
) AS unique_songs
GROUP BY success_tier
ORDER BY avg_danceability DESC
"""

result_2020 = pd.read_sql(query_2020, con=engine)
display(result_2020)

,success_tier,avg_danceability,avg_energy,avg_tempo,avg_valence,num_songs
0,Charted outside top 40,0.669514,0.623880,123.525447,0.471662,432
1,Top 10,0.660513,0.589037,127.574387,0.504032,80
2,Top 40,0.660257,0.593178,119.172942,0.480220,191


**Takeaway:** In the 2020s, the danceability pattern reverses — "Charted Outside Top 40" songs (0.670) now show *higher* average danceability than Top 10 hits (0.661). This suggests danceability may no longer differentiate hits from non-hits, as it has become a baseline expectation across nearly all competitive tracks. 

*Caveat: the Top 10 sample size for this period (n=80) is small relative to earlier decades, so this finding should be treated as directional rather than conclusive.*

### 6d. Decade-by-Decade Comparison (2000s, 2010s, 2020s)

In [84]:
# Compare success-tier audio feature averages across the 2000s, 2010s, and 2020s
query_decades = """
SELECT 
    decade,
    CASE 
        WHEN highest_position <= 10 THEN 'Top 10'
        WHEN highest_position <= 40 THEN 'Top 40'
        ELSE 'Charted outside top 40'
    END AS success_tier,
    AVG(danceability) AS avg_danceability,
    AVG(energy) AS avg_energy,
    AVG(tempo) AS avg_tempo,
    AVG(valence) AS avg_valence,
    COUNT(*) AS num_songs
FROM (
    SELECT 
        SongID,
        MIN("Peak Position") AS highest_position,
        danceability,
        energy,
        tempo,
        valence,
        (CAST(SUBSTR(WeekID, -4) AS INTEGER) / 10) * 10 AS decade
    FROM billboard_history
    WHERE danceability IS NOT NULL
    AND (CAST(SUBSTR(WeekID, -4) AS INTEGER) / 10) * 10 IN (2000, 2010, 2020)
    GROUP BY SongID, danceability, energy, tempo, valence, decade
) AS unique_songs
GROUP BY decade, success_tier
ORDER BY decade, success_tier
"""

result_decades = pd.read_sql(query_decades, con=engine)
display(result_decades)

,decade,success_tier,avg_danceability,avg_energy,avg_tempo,avg_valence,num_songs
0,2000,Charted outside top 40,0.605318,0.721985,122.656372,0.538015,1649
1,2000,Top 10,0.672702,0.702852,118.301963,0.581966,567
2,2000,Top 40,0.627467,0.712274,118.698756,0.556209,998
3,2010,Charted outside top 40,0.628031,0.666918,124.057016,0.470978,2706
4,2010,Top 10,0.671943,0.677848,120.681720,0.519207,507
5,2010,Top 40,0.640532,0.669697,122.898690,0.480043,1105
6,2020,Charted outside top 40,0.669514,0.623880,123.525447,0.471662,432
7,2020,Top 10,0.660513,0.589037,127.574387,0.504032,80
8,2020,Top 40,0.660257,0.593178,119.172942,0.480220,191


**Takeaway:** The danceability gap between Top 10 hits and lower-charting songs has shrunk steadily each decade — from 0.067 in the 2000s, to 0.044 in the 2010s, to just 0.005 in the 2020s. Energy has also declined across all tiers over time. This suggests danceability, once a meaningful differentiator, has become table-stakes for any track competing for playlist placement — no longer distinguishing hits from non-hits the way it once did.

## Key Findings & Implications for Scouting

1. Danceability's role in separating hits from non-hits has weakened significantly — from a 0.067 gap (2000s) to just 0.005 (2020s).
2. Valence (musical positivity) dropped notably after the 2000s and has remained lower since, reflecting a broader shift toward moodier mainstream production.
3. **Implication:** Static audio characteristics are no longer sufficient to predict hit potential in today's market. Modern A&R scouting requires momentum-based signals — streaming growth trajectory, playlist placement, and social traction — explored in Part 2 of this project.